In [ ]:

!pip install -q google-genai pydantic

import pandas as pd
import json
import time
from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

API_KEY = getpass("Enter API key")
client = genai.Client(api_key=API_KEY)
del API_KEY

MODEL_NAME = "gemini-3.8-flash"
print("Gemini client configured!")


gen-lang-client-0732257442··········
Gemini client configured!


## Task 4.1: Build the AI classification function

In [ ]:

ALLOWED = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause", "confidence",
        "summary", "recommended_action"
    ]
}

SYSTEM_TEXT = (
    "You classify outpatient appointment no-show notes. "
    "Do not invent facts. Return valid JSON only."
)

api_calls = 0

def analyse_note(note_text, version="v2"):
    global api_calls

    if version == "v1":
        prompt = f"Classify this note:\n{note_text}"
    else:
        prompt = f"""
Context: A patient missed a clinic appointment.
Role: Healthcare operations analyst.
Action: Identify the main reason for missing the visit.
Allowed root_cause values: {", ".join(ALLOWED)}
Use 'other' when the reason is unclear.
Tone: Neutral, concise, factual.
Format: JSON containing root_cause, confidence (0-1),
summary (maximum 25 words), and recommended_action.

Patient note:
{note_text}
"""

    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw = response.text
    return json.loads(raw), raw


In [ ]:

notes = pd.read_csv("/content/Follow_up notes csv.csv")

note1 = notes.loc[
    notes["note_id"] == "N001", "note_text"
].iloc[0]

result, raw = analyse_note(note1)

print(json.dumps(result, indent=2))


{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "Patient forgot the appointment due to not receiving a reminder message and requested to rebook next week.",
  "recommended_action": "Verify patient contact details to ensure reminder delivery and reschedule the appointment for next week."
}


## Task 4.2: Validate and analyze 10 notes


In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

results = []

for _, row in notes.head(10).iterrows():
    note_text = row["note_text"]

    record = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"]
    }

    if pd.isna(note_text) or str(note_text).strip() == "":
        record.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
    else:
        raw = ""
        data = {}

        try:
            data, raw = analyse_note(str(note_text))
            validated = FollowupInsight(**data)

            if validated.root_cause not in ALLOWED:
                raise ValueError("Invalid root cause")

            status = "valid"

        except Exception as e:
            print(row["note_id"], "Error:", e)
            status = "invalid"

        record.update({
            "root_cause": data.get("root_cause"),
            "confidence": data.get("confidence"),
            "summary": data.get("summary"),
            "recommended_action": data.get(
                "recommended_action"
            ),
            "validation_status": status,
            "raw_response": raw
        })

        time.sleep(13)

    results.append(record)

output_cols = [
    "note_id", "appointment_id", "root_cause",
    "confidence", "summary", "recommended_action",
    "validation_status", "raw_response"
]

notes_df = pd.DataFrame(results)[output_cols]
notes_df.to_csv("notes_analysed.csv", index=False)

print(notes_df[[
    "note_id", "root_cause", "validation_status"
]])
print("Shape:", notes_df.shape)
print("API calls:", api_calls)


N003 Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N010 Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
  note_id             root_cause validation_status
0    N001  forgot_or_no_reminder             valid
1    N002  distance_or_transport             valid
2    N003                   None           invalid
3    N004                  other     skipped_blank
4    N005   illness_or_emergency             valid
5    N006       work_or_schedule             valid
6    N007  forgot_or_no_reminder             valid
7    N008  distance_or_transport             valid
8    N009                  other             valid
9    N010                   None           invalid
Shape: (10, 8)
API calls:

## Task 4.3: Test validation failures

In [ ]:

bad_replies = [
    {
        "root_cause": "other",
        "confidence": 1.7,
        "summary": "Patient reason unclear",
        "recommended_action": "Call again"
    },
    {
        "root_cause": "cost_or_fee",
        "confidence": 0.9,
        "summary": "Patient mentioned high fees"
    }
]

for reply in bad_replies:
    try:
        FollowupInsight(**reply)
    except ValidationError as e:
        print("Validation correctly failed:")
        print(e)


Validation correctly failed:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Validation correctly failed:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...nt mentioned high fees'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:




prompt_tests = []

for note_id in ["N007", "N009"]:
    note = notes.loc[
        notes["note_id"] == note_id, "note_text"
    ].iloc[0]

    for version in ["v1", "v2"]:
        time.sleep(13)

        try:
            data, raw = analyse_note(note, version)
            cause = data.get("root_cause", "")
            valid = cause in ALLOWED
        except Exception as e:
            cause = str(e)
            valid = False

        prompt_tests.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": cause,
            "allowed_value": valid
        })

prompt_df = pd.DataFrame(prompt_tests)
print(prompt_df)
print("Total API calls:", api_calls)


KeyboardInterrupt: 

## Task 4.4: Merge AI results with appointments

In [ ]:

appointments = pd.read_csv("appointments_clean.csv")

merged = notes_df.merge(
    appointments,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

print("Merged rows:", len(merged))

cross_tab = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)

print(cross_tab)
print("\nRoot cause counts:")
print(notes_df["root_cause"].value_counts())


NameError: name 'notes_df' is not defined